# Data

In [4]:
import pandas as pd
import numpy as np


df = pd.read_json("train.jsonl", lines=True)

df.head()

,id,text,model,label,genre
0,ea468d03-1973-5039-86b2-ff225bb92c4e,"Duke Ellington, a titan of jazz, revolutionize...",falcon3-10b-instruct,1,essays
1,0d05f269-6d67-521d-9b5d-cc18f482c6c1,I reflected on the shifting dynamics of media ...,o3-mini,1,essays
2,c2ec79f3-da80-58f8-bef0-3e0ea7ab072f,"In F. Scott Fitzgerald's ""The Great Gatsby,"" t...",gpt-4o,1,essays
3,4ad37c58-0bb7-536b-997d-cfccabd0d094,I still chuckle when I think about that time I...,deepseek-r1-distill-qwen-32b,1,essays
4,07747b0c-5051-5e0d-8096-b4d4ed8bd98e,"Yoga, originating in ancient India, encompasse...",gemini-2.0-flash,1,essays


In [5]:
import fasttext
import fasttext.util
from pathlib import Path

model_directory = Path("models").resolve()

# Korrekt download-kald til din installerede version
downloaded_path = fasttext.util.download_model(
    category="vectors_crawl",
    lang_id="en",
    model_type="bin",
    save_dir=str(model_directory),
    if_exists="ignore",
)

# Stop tydeligt, hvis download mislykkes
if downloaded_path is None:
    raise RuntimeError(
        "Modellen blev ikke downloadet. Se fejlmeddelelsen ovenfor."
    )

model_path = Path(downloaded_path).resolve()

if not model_path.is_file():
    raise FileNotFoundError(f"Modelfilen findes ikke: {model_path}")

print(f"Indlæser modellen fra: {model_path}")
ft_model = fasttext.load_model(str(model_path))

print(f"Embedding-dimension: {ft_model.get_dimension()}")

Found URL for download: https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz


cc.en.300.bin.gz: 100%|██████████| 4.50G/4.50G [04:34<00:00, 16.4MB/s]


Decompressing 'cc.en.300.bin.gz'...
Indlæser modellen fra: C:\Users\Daniel K\Desktop\DAKI\5. Semester\mini-projekt\NLP-mini-project\models\cc.en.300.bin
Embedding-dimension: 300


In [6]:
def create_fasttext_embeddings(df, model, text_column="text"):
    texts = df[text_column].fillna("")
    dimension = model.get_dimension()

    # Én række pr. tekst og én kolonne pr. embedding-dimension
    vectors = np.zeros(
        (len(df), dimension),
        dtype=np.float32,
    )

    for position, text in enumerate(
        tqdm(texts, total=len(texts), desc="Beregner embeddings")
    ):
        # fastText forventer én tekstlinje.
        # Saml afsnit, linjeskift og anden whitespace.
        clean_text = " ".join(text.split())

        if clean_text:
            vectors[position] = model.get_sentence_vector(clean_text)

    return pd.DataFrame(
        vectors,
        index=df.index,
        columns=[f"fasttext_{i:03d}" for i in range(dimension)],
    )


# Beregn embeddings
embeddings_df = create_fasttext_embeddings(df, ft_model)

print("Embeddings-form:", embeddings_df.shape)
embeddings_df.head()

Beregner embeddings:   0%|          | 0/23707 [00:00<?, ?it/s]

Embeddings-form: (23707, 300)


,fasttext_000,fasttext_001,fasttext_002,fasttext_003,fasttext_004,fasttext_005,fasttext_006,fasttext_007,fasttext_008,fasttext_009,...,fasttext_290,fasttext_291,fasttext_292,fasttext_293,fasttext_294,fasttext_295,fasttext_296,fasttext_297,fasttext_298,fasttext_299
0,-0.008662,-0.010640,0.012477,0.034961,-0.019263,-0.010210,0.015301,-0.004503,-0.006636,0.004243,...,0.032025,-0.007812,-0.048761,0.011722,-0.013149,-0.009398,-0.008655,0.063485,-0.018474,0.010357
1,-0.017968,0.006717,0.009445,0.041145,-0.020060,-0.002876,0.012798,-0.009374,-0.012614,0.004530,...,0.028453,-0.008466,-0.052252,0.013798,-0.008793,-0.006192,-0.012222,0.072317,-0.003214,0.022588
2,-0.013407,-0.010420,0.007419,0.030161,-0.018619,-0.017242,0.015260,-0.004623,-0.007839,0.004898,...,0.026562,-0.018444,-0.040815,0.011698,-0.012334,-0.014555,-0.005281,0.061844,-0.017277,0.016748
3,-0.005556,-0.010932,0.002127,0.040707,-0.019978,-0.010869,0.008636,0.004716,-0.006299,0.000845,...,0.035286,-0.003402,-0.068106,-0.000909,-0.013418,-0.003034,0.007204,0.075372,-0.014579,-0.004184
4,-0.030250,-0.003060,0.009320,0.038593,-0.020043,-0.031695,0.016456,0.012787,-0.008463,-0.000436,...,0.030857,-0.001724,-0.039564,0.007523,-0.014244,-0.015827,0.005745,0.059706,-0.013571,0.020166


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Kontrollér, at embeddings og labels følger samme rækkeindeks
if not embeddings_df.index.equals(df.index):
    raise ValueError(
        "embeddings_df og dataset skal have samme rækkeindeks "
        "i samme rækkefølge."
    )

X_fasttext = embeddings_df.copy()
y_fasttext = df["label"].copy()

if not y_fasttext.isin([0, 1]).all() or y_fasttext.nunique() != 2:
    raise ValueError("label skal indeholde begge grupper: 0 og 1, uden NaN.")

# Samme split-indstillinger som i de tidligere modeller
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(
    X_fasttext,
    y_fasttext,
    test_size=0.20,
    random_state=42,
    stratify=y_fasttext,
)

# Imputering og skalering tilpasses kun på træningsdata
logistic_model_fasttext = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
])

logistic_model_fasttext.fit(X_train_ft, y_train_ft)

# Forudsigelser på testdata
y_pred_ft = logistic_model_fasttext.predict(X_test_ft)
y_probability_ft = logistic_model_fasttext.predict_proba(X_test_ft)[:, 1]

# Evaluering
scores_fasttext = {
    "Accuracy": accuracy_score(y_test_ft, y_pred_ft),
    "Balanced accuracy": balanced_accuracy_score(y_test_ft, y_pred_ft),
    "ROC-AUC": roc_auc_score(y_test_ft, y_probability_ft),
}

print(f"Træningstekster: {len(X_train_ft)}")
print(f"Testtekster:    {len(X_test_ft)}")
print(f"Embedding-dimensioner: {X_fasttext.shape[1]}")

print("\nResultater — fastText og logistisk regression:")
print(pd.Series(scores_fasttext).round(3).to_string())

print("\nClassification report:")
print(classification_report(
    y_test_ft,
    y_pred_ft,
    labels=[0, 1],
    target_names=["Menneske", "Maskine"],
    digits=3,
    zero_division=0,
))

print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test_ft, y_pred_ft, labels=[0, 1]),
    index=["Faktisk Menneske", "Faktisk Maskine"],
    columns=["Forudsagt Menneske", "Forudsagt Maskine"],
))

Træningstekster: 18965
Testtekster:    4742
Embedding-dimensioner: 300

Resultater — fastText og logistisk regression:
Accuracy             0.979
Balanced accuracy    0.979
ROC-AUC              0.995

Classification report:
              precision    recall  f1-score   support

    Menneske      0.966     0.979     0.972      1820
     Maskine      0.987     0.978     0.983      2922

    accuracy                          0.979      4742
   macro avg      0.976     0.979     0.978      4742
weighted avg      0.979     0.979     0.979      4742

Confusion matrix:
                  Forudsagt Menneske  Forudsagt Maskine
Faktisk Menneske                1782                 38
Faktisk Maskine                   63               2859


: 